In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import h5py, os, yaml, pickle
import numpy as np
import healpy as hp
import matplotlib.pyplot as plt
from tqdm import tqdm

import tensorflow as tf
tf.config.experimental.set_memory_growth(tf.config.list_physical_devices(device_type="GPU")[0], True)

from deep_lss.models.base_model import BaseModel
from deep_lss.utils import configuration
from deep_lss.nets import NETWORKS
from msi.utils import preprocessing
from msfm.utils import prior, parameters, files, logger, observation

# v9

### clustering

In [3]:
# msfm_conf = files.load_config("/global/homes/a/athomsen/multiprobe-simulation-forward-model/configs/v9/linear_bias.yaml")
# base_dir = "/pscratch/sd/a/athomsen/run_files/v9/clustering"

In [4]:
# # earnest-wood-1039 (https://wandb.ai/eth-cosmo/y3-deep-lss/runs/nmhf2f7f), 10% of noise
# model_dir = "delta/2024-07-17_09-27-28_resnet_vanilla"
# net_conf = "/global/u2/a/athomsen/y3-deep-lss/configs/v9/clustering/resnet_vanilla.yaml"
# dlss_conf = configuration.load_deep_lss_config(
#     "/global/u2/a/athomsen/y3-deep-lss/configs/v9/clustering/linear_bias/dlss_config_10.yaml"
# )
# n_steps = 60_000

In [5]:
# # fresh-sky-1040 (https://wandb.ai/eth-cosmo/y3-deep-lss/runs/0r0mzpo9), 1% of noise
# model_dir = "delta/2024-07-18_10-16-57_resnet_vanilla"
# net_conf = "/global/u2/a/athomsen/y3-deep-lss/configs/v9/clustering/resnet_vanilla.yaml"
# dlss_conf = configuration.load_deep_lss_config(
#     "/global/u2/a/athomsen/y3-deep-lss/configs/v9/clustering/linear_bias/dlss_config.yaml"
# )
# n_steps = 60_000

# v10

### lensing

In [6]:
# conf = files.load_config("/global/homes/a/athomsen/multiprobe-simulation-forward-model/configs/v9/linear_bias.yaml")
# base_dir = "/pscratch/sd/a/athomsen/run_files/v10/lensing"

# # mutual info loss ###############################################################################################

# # # eternal-tree-1042 (https://wandb.ai/eth-cosmo/y3-deep-lss/runs/rh93pfh8/overview) 
# # # default
# # model_dir = "mutual_info/2024-08-26_06-07-27_deepsphere_default"
# # net_conf = "/global/u2/a/athomsen/y3-deep-lss/configs/v10/lensing/deepsphere_default.yaml"
# # n_steps = 100_000

# # dulcet-dragon-1048 (https://wandb.ai/eth-cosmo/y3-deep-lss/runs/xhb0e1xg/overview)
# # flat learning rate instead of cosine decay
# model_dir = "mutual_info/2024-08-28_00-40-10_deepsphere_default"
# net_conf = "/global/u2/a/athomsen/y3-deep-lss/configs/v10/lensing/deepsphere_default.yaml"
# n_steps = 100_000

# # fisher info loss ###############################################################################################

# # # # valiant-forest-1041 (https://wandb.ai/eth-cosmo/y3-deep-lss/runs/na3hr7y4/overview)
# # model_dir = "delta/2024-08-26_06-07-05_deepsphere_default"
# # net_conf = "/global/u2/a/athomsen/y3-deep-lss/configs/v10/lensing/deepsphere_default.yaml"
# # n_steps = 100_000

# # clean-sunset-1045 (https://wandb.ai/eth-cosmo/y3-deep-lss/runs/9z04yyv8/overview)

# params = ["Om", "s8", "w0", "Aia", "n_Aia"]

### clustering

In [7]:
# base_dir = "/pscratch/sd/a/athomsen/run_files/v10/clustering"

# # mutual info loss ###############################################################################################

# # lyric-cosmos-1053 (https://wandb.ai/eth-cosmo/y3-deep-lss/runs/vlqubdzi/overview)
# # smaller batch size and more gradient clipping
# # this does not work 
# loss_func = "mutual"
# model_dir = "mutual_info/2024-08-29_02-37-49_deepsphere_default"
# model_dir = os.path.join(base_dir, model_dir)
# # msfm_conf = files.load_config("/global/homes/a/athomsen/multiprobe-simulation-forward-model/configs/v10/linear_bias.yaml")
# # net_conf = "/global/u2/a/athomsen/y3-deep-lss/configs/v10/clustering/deepsphere_default.yaml"
# # dlss_conf = "/global/u2/a/athomsen/y3-deep-lss/configs/v10/clustering/dlss_config.yaml"
# n_summary = 10
# n_steps = 190_000
# mocks = ["Buzzard", "Cardinal"]

# # # rich-dust-1071 (https://wandb.ai/eth-cosmo/y3-deep-lss/runs/adhedyyr/overview)
# # loss_func = "mutual"
# # model_dir = "/pscratch/sd/a/athomsen/run_files/v10/linear_bias_octant/clustering/mutual_info/2024-09-08_10-01-44_deepsphere_default"
# # # msfm_conf = files.load_config("/global/homes/a/athomsen/multiprobe-simulation-forward-model/configs/v10/linear_bias.yaml")
# # # dlss_conf = "/global/u2/a/athomsen/y3-deep-lss/configs/v10/clustering/dlss_config.yaml"
# # # net_conf = "/global/u2/a/athomsen/y3-deep-lss/configs/v10/clustering/deepsphere_default.yaml"
# # n_summary = 10
# # n_steps = 140_000
# # mocks = ["MICE"]

# # fisher info loss ###############################################################################################

# # # astral-snowball-1050 (https://wandb.ai/eth-cosmo/y3-deep-lss/runs/165eicwm/overview)
# # # this works
# # loss_func = "delta"
# # model_dir = "/pscratch/sd/a/athomsen/run_files/v10/clustering/delta/2024-08-28_02-52-15_deepsphere_default"
# # net_conf = "/global/u2/a/athomsen/y3-deep-lss/configs/v10/clustering/deepsphere_no_dropout.yaml"
# # dlss_conf = "/global/u2/a/athomsen/y3-deep-lss/configs/v10/clustering/dlss_config.yaml"
# # n_summary = 5
# # n_steps = 70_000
# # mocks = ["Buzzard", "Cardinal"]

# # net_conf = configuration.load_deep_lss_config(net_conf)
# # dlss_conf = configuration.load_deep_lss_config(dlss_conf)

# with open(os.path.join(model_dir, "configs.yaml"), "r") as f:
#     net_conf, dlss_conf, msfm_conf = list(yaml.load_all(f, Loader=yaml.FullLoader))
# params = ["Om", "s8", "w0", "bg", "n_bg"]
# n_z = 4
# data_vec_pix = files.load_pixel_file(msfm_conf)[0]

# v11

In [8]:
# base_dir = ""

# # mutual info loss ###############################################################################################
# loss_func = "mutual"

# # # sunny-sea-1078 (https://wandb.ai/eth-cosmo/y3-deep-lss/runs/kt4ng005/overview)
# # # soft cut, but similar l_max as earnest-flower-1077
# # model_dir = "/pscratch/sd/a/athomsen/run_files/v11/extended/clustering/mutual_info/2024-10-12_04-01-53_deepsphere_default"
# # n_steps = 130_000

# # # earnest-flower-1077 (https://wandb.ai/eth-cosmo/y3-deep-lss/runs/7nl9qkh8/overview)
# # # hard cut, but similar l_max as sunny-sea-1078
# # model_dir = "/pscratch/sd/a/athomsen/run_files/v11/extended_hard_cut/clustering/mutual_info/2024-10-12_04-01-53_deepsphere_default"
# # n_steps = 200_000

# # vague-sea-1080 (https://wandb.ai/eth-cosmo/y3-deep-lss/runs/kt4ng005/overview)
# # soft cut with a lot of smoothing
# model_dir = "/pscratch/sd/a/athomsen/run_files/v11/extended/clustering/mutual_info/2024-10-20_01-03-44_deepsphere_default"
# n_steps = 110_000

# # fisher info loss ###############################################################################################

# with open(os.path.join(model_dir, "configs.yaml"), "r") as f:
#     net_conf, dlss_conf, msfm_conf = list(yaml.load_all(f, Loader=yaml.FullLoader))

# n_z = 4
# data_vec_pix = files.load_pixel_file(msfm_conf)[0]
# params = ["Om", "s8", "w0", "bg1", "bg2", "bg3", "bg4"]
# mocks = ["Buzzard", "Buzzard_flock", "Cardinal"]
# # mocks = ["Buzzard", "Cardinal"]

# v12

In [9]:
# base_dir = ""

# # mutual info loss ###############################################################################################
# loss_func = "mutual"

# # fanciful-plasma-1081 (https://wandb.ai/eth-cosmo/y3-deep-lss/runs/xus25zc2/overview)
# # very conservative like vague-sea-1080, but with (DeepLSS-style) quadratic biasing
# model_dir = "/pscratch/sd/a/athomsen/run_files/v12/nonlinear/clustering/mutual_info/2024-10-26_18-21-39_deepsphere_default"
# n_steps = 100_000

# # fisher info loss ###############################################################################################

# with open(os.path.join(model_dir, "configs.yaml"), "r") as f:
#     net_conf, dlss_conf, msfm_conf = list(yaml.load_all(f, Loader=yaml.FullLoader))

# n_z = 4
# data_vec_pix = files.load_pixel_file(msfm_conf)[0]
# params = ["Om", "s8", "w0", "bg1", "bg2", "bg3", "bg4", "qbg1", "qbg2", "qbg3", "qbg4"]
# mocks = ["Buzzard", "Buzzard_flock", "Cardinal"]
# # mocks = ["Buzzard", "Cardinal"]

# v13

### lensing

In [10]:
# # worldly-cherry-1082 (https://wandb.ai/eth-cosmo/y3-deep-lss/runs/axisu9uv/overview)
# # conservative first v13 run
# model_dir = "/pscratch/sd/a/athomsen/run_files/v13/extended/lensing/mutual_info/2025-01-11_07-17-28_deepsphere_default"
# # n_steps = 100_000
# n_steps = 200_000

# params = ["Om", "s8", "w0", "Aia", "n_Aia", "bta"]

### clustering

In [11]:
# # # # polished-snow-1083 (https://wandb.ai/eth-cosmo/y3-deep-lss/runs/8bptajqu/overview)
# # # # very conservative first v13 run
# # # model_dir = "/pscratch/sd/a/athomsen/run_files/v13/extended/clustering/mutual_info/2025-01-11_07-17-28_deepsphere_default"
# # # n_steps = 100_000

# # # wandering-shadow-1084 (https://wandb.ai/eth-cosmo/y3-deep-lss/runs/z9to945a/overview)
# # # even more conservative run, 32mpc/h smoothing with 10% noise -> l_max = [82, 121, 158, 189]
# # model_dir = "/pscratch/sd/a/athomsen/run_files/v13/extended/clustering/mutual_info/2025-01-16_00-50-58_deepsphere_default"
# # # n_steps = 80_000
# # # n_steps = 140_000
# # # n_steps = 120_000
# # n_steps = 170_000

# # confused-sun-1087 (https://wandb.ai/eth-cosmo/y3-deep-lss/runs/ekt8evre/overview)
# # less conservative run, 32mpc/h smoothing with 1% noise -> l_max = [133, 195, 255, 305]
# model_dir = "/pscratch/sd/a/athomsen/run_files/v13/extended/clustering/mutual_info/2025-02-01_01-16-48_deepsphere_default"
# # n_steps = 110_000
# # n_steps = 140_000
# n_steps = 190_000

# params = ["Om", "s8", "w0", "bg1", "bg2", "bg3", "bg4"]

# v14

### lensing

In [12]:
# honest-haze-1091 (https://wandb.ai/eth-cosmo/y3-deep-lss/runs/58tev0dz/overview)
# first v14 run
model_dir = "/pscratch/sd/a/athomsen/run_files/v14/extended/lensing/mutual_info/2025-04-19_18-54-31_deepsphere_default"
# n_steps = 100_000
# n_steps = 200_000
n_steps = 300_000
# n_steps = 400_000

checkpoint_number = n_steps // 10_000
# checkpoint_number = 21

params = ["Om", "s8", "w0", "Aia", "n_Aia", "bta"]

### clustering

In [13]:
# # # wandering-shadow-1084 (https://wandb.ai/eth-cosmo/y3-deep-lss/runs/z9to945a/overview)
# # # even more conservative run, 32mpc/h smoothing with 10% noise -> l_max = [82, 121, 158, 189]
# # model_dir = "/pscratch/sd/a/athomsen/run_files/v13/extended/clustering/mutual_info/2025-01-16_00-50-58_deepsphere_default"
# # # n_steps = 80_000
# # # n_steps = 140_000
# # # n_steps = 120_000
# # n_steps = 170_000

# params = ["Om", "s8", "w0", "bg1", "bg2", "bg3", "bg4"]

In [14]:
base_dir = ""

# mutual info loss ###############################################################################################
loss_func = "mutual"

# fisher info loss ###############################################################################################

with open(os.path.join(model_dir, "configs.yaml"), "r") as f:
    net_conf, dlss_conf, msfm_conf = list(yaml.load_all(f, Loader=yaml.FullLoader))

n_z = 4
data_vec_pix = files.load_pixel_file(msfm_conf)[0]
# mocks = ["Buzzard", "Buzzard_flock", "Cardinal"]
# mocks = ["Buzzard", "Cardinal"]
mocks = ["Buzzard"]

25-05-12 06:55:26     files.py INF   Loaded the pixel file /global/u2/a/athomsen/multiprobe-simulation-forward-model/data/DESY3_pixels_v11_fiducial_512.h5 


# build the network

In [15]:
smoothing_kwargs = configuration.get_smoothing_kwargs(
    loss_func, msfm_conf, dlss_conf, net_conf, dir_base="."
)
# smoothing_kwargs = None
# smoothing_kwargs["white_noise_sigma"] = None
smoothing_kwargs["max_batch_size"] = 1

# size of the summary statistic, depends on the loss function
out_features =  2 * len(params)
# out_features =  len(params)
# out_features = n_summary

# load the layers
network = NETWORKS[net_conf["network"]["name"]](
    out_features=out_features, smoothing_kwargs=smoothing_kwargs, **net_conf["network"]["kwargs"]
).get_layers()

# build the model, same regardless of the loss function (fiducial or grid)
model = BaseModel(
    network=network,
    n_side=msfm_conf["analysis"]["n_side"],
    indices=data_vec_pix,
    # n_neighbors=net_conf["network"]["n_neighbors"],
    input_shape=(None, len(data_vec_pix), n_z),
    checkpoint_dir=os.path.join(base_dir, model_dir, "checkpoint"),
    max_batch_size=64,
    # max_batch_size=None,
    # always load from a checkpoint
    # restore_checkpoint=True,
    restore_checkpoint=False,
    strategy=None,
)

model.restore_model_from_checkpoint_dir(
    os.path.join(base_dir, model_dir, "checkpoint", f"ckpt-{checkpoint_number}.index")
)

25-05-12 06:55:26     files.py INF   Loaded the pixel file /global/u2/a/athomsen/multiprobe-simulation-forward-model/data/DESY3_pixels_v11_fiducial_512.h5 
25-05-12 06:55:26     files.py INF   Loaded the pixel file /global/u2/a/athomsen/multiprobe-simulation-forward-model/data/DESY3_pixels_v11_fiducial_512.h5 
Using the per channel smoothing repetitions [6 3 2 1]
Using the per channel smoothing scales sigma = [13.11  9.27  7.57  5.35] arcmin, fwhm = [30.86 21.82 17.82 12.6 ] arcmin
Successfully loaded sparse kernel indices and values from ./smoothing
Successfully created the sparse kernel tensor
Adding white noise with sigma [0.03520349 0.02952409 0.02803563 0.02272882] to the smoothed map. Note that networks outputs are nondeterministic, even for training=False
25-05-12 06:55:29 regression_h INF   Using a dense regression head 
25-05-12 06:55:29 regression_h WAR   Using dropout with probability 0.01 in the regression head 
25-05-12 06:55:29 base_model.p INF   Initializing with a Healp

In [16]:
# model.restore_model_from_checkpoint_dir(
#     "/pscratch/sd/a/athomsen/run_files/v14/extended/lensing/mutual_info/2025-04-19_18-54-31_deepsphere_default/checkpoint/ckpt-30.index"
# )

# forward model the mock

In [17]:
out_file = os.path.join(base_dir, model_dir, f"preds_{n_steps}.h5")
print(out_file)

/pscratch/sd/a/athomsen/run_files/v14/extended/lensing/mutual_info/2025-04-19_18-54-31_deepsphere_default/preds_300000.h5


### weak lensing

#### CosmoGrid

In [48]:
# buzzard_lensing_file = "/pscratch/sd/j/jbucko/DESY3/mock_observations/lensing/buzzard_flock/0/DESY3_mock_observation_buzzard_0_shape_noise_fw_g1g2_Buzzard.h5"
# buzzard_lensing_file = "/pscratch/sd/j/jbucko/DESY3/mock_observations/lensing/buzzard_flock/0/DESY3_mock_observation_buzzard_0_shape_noise_fw_g1g2_Buzzard_shape_noise_from_cosmogrid.h5"
# buzzard_lensing_file = "/pscratch/sd/j/jbucko/DESY3/mock_observations/lensing/buzzard_flock/0/DESY3_mock_observation_buzzard_0_shape_noise_fw_g1g2_Buzzard_shape_noise_from_cosmogrid_no_true_shear.h5"

# buzzard_lensing_file = "/pscratch/sd/j/jbucko/DESY3/mock_observations/lensing/buzzard_flock/0/DESY3_mock_observation_buzzard_0_shape_noise_from_biases_fitted_to_cosmogrid_g1g2_true_from_Buzzard_fixed_ordering.h5"

# v14
# buzzard_lensing_file = "/pscratch/sd/j/jbucko/DESY3/mock_observations/lensing/buzzard_flock/0/DESY3_mock_observation_buzzard_flock_v14_shear_noise+WL.h5"
# buzzard_lensing_file = "/pscratch/sd/j/jbucko/DESY3/mock_observations/lensing/buzzard_flock/5/DESY3_mock_observation_buzzard_flock_v14_shear_noise+WL.h5"
# buzzard_lensing_file = "/pscratch/sd/j/jbucko/DESY3/mock_observations/lensing/buzzard_flock/8/DESY3_mock_observation_buzzard_flock_v14_shear_noise+WL.h5"
buzzard_lensing_file = "/pscratch/sd/j/jbucko/DESY3/mock_observations/lensing/buzzard_flock/11/DESY3_mock_observation_buzzard_flock_v14_shear_noise+WL.h5"

with h5py.File(buzzard_lensing_file, "r") as f_in:
    gamma1 = []
    gamma2 = []
    for i in range(1,5):
        gamma1.append(f_in[f"metacal/raw_gamma1_bin{i}"])
        gamma2.append(f_in[f"metacal/raw_gamma2_bin{i}"])
    gamma1 = np.stack(gamma1, axis=-1)
    gamma2 = np.stack(gamma2, axis=-1)
    
    wl_map = np.stack([gamma1, gamma2], axis=-1)

obs_map, _, _ = observation.forward_model_observation_map(
    wl_gamma_map=wl_map,
    conf=msfm_conf,
    apply_norm=True,
    with_padding=True,
    # nest_in=True,
    nest_in=False,
)

with h5py.File(out_file, "a") as f_out:
    obs_pred = model(obs_map[np.newaxis], training=False)
    obs_pred = np.squeeze(obs_pred)

    # out_label = f"mocks/pred/Buzzard_0"
    # out_label = f"mocks/pred/Buzzard_5"
    # out_label = f"mocks/pred/Buzzard_8"
    out_label = f"mocks/pred/Buzzard_11"

    if out_label in f_out:
        del f_out[out_label]
    f_out.create_dataset(name=out_label, data=obs_pred)

25-05-12 06:57:18     files.py INF   Loaded the pixel file /global/u2/a/athomsen/multiprobe-simulation-forward-model/data/DESY3_pixels_v11_fiducial_512.h5 
25-05-12 06:57:18     files.py INF   Loaded the pixel file /global/u2/a/athomsen/multiprobe-simulation-forward-model/data/DESY3_pixels_v11_fiducial_512.h5 
25-05-12 06:57:18     files.py INF   Loaded the pixel file /global/u2/a/athomsen/multiprobe-simulation-forward-model/data/DESY3_pixels_v11_fiducial_512.h5 
25-05-12 06:57:19     files.py INF   Loaded the pixel file /global/u2/a/athomsen/multiprobe-simulation-forward-model/data/DESY3_pixels_v11_fiducial_512.h5 


In [49]:
obs_pred

array([-1.3375542 ,  0.62076235,  0.19473691, -1.9660277 , -0.6561922 ,
       -1.3991268 , -1.3537859 ,  1.0504827 , -1.3809534 , -1.5673393 ,
        0.22283797,  0.8440041 ], dtype=float32)

In [50]:
with h5py.File(out_file, "r") as f_out:
    print(f_out["mocks/pred"].keys())

<KeysViewHDF5 ['Buzzard', 'Buzzard_0', 'Buzzard_11', 'Buzzard_5', 'Buzzard_8']>


In [21]:
# # get cosmogrid cosmologies
# data = h5py.File('/global/cfs/cdirs/des/cosmogrid/CosmoGridV1_metainfo.h5','r')
# cosmo_arr = data['parameters/grid'][()]

# grid_cosmo_paths = [item.astype('str').split('/')[-2] for item in cosmo_arr['path_par']]
# cosmo_params_cosmogridv11 = np.c_[cosmo_arr['Om'].astype('float'),cosmo_arr['s8'].astype('float'),cosmo_arr['w0'].astype('float'),cosmo_arr['wa'].astype('float'),cosmo_arr['Ob'].astype('float'),cosmo_arr['ns'].astype('float'),cosmo_arr['H0'].astype('float')/100]

# # select some cosmologies
# cosmogrid_grid_idx = [0,38,654,710]

# # cosmo_params_cosmogridv11[cosmogrid_grid_idx]

In [22]:
# lensing_file = "/pscratch/sd/j/jbucko/DESY3/v13/linear_bias/lensing/maps/maps_for_compression_v13_clustering_False_lensing_True_grid_cosmo.pkl"
# with open(lensing_file, 'rb') as f:
#     jozef_maps = pickle.load(f)

# n_side = msfm_conf["analysis"]["n_side"]
# n_pix = msfm_conf["analysis"]["n_pix"]
# data_vec_pix = files.load_pixel_file(msfm_conf)[0]
# hp_datapath = "/global/u2/a/athomsen/multiprobe-simulation-forward-model/data/healpy_data"

# with h5py.File(out_file, "a") as f_out:
#     for i in range(jozef_maps.shape[0]):
#         j = cosmogrid_grid_idx[i]
        
#         wl_map = jozef_maps[i]
#         print(wl_map.shape)

#         obs_pred = model(wl_map, training=False)
#         obs_pred = np.squeeze(obs_pred)

#         # print("\n", obs_label, obs_pred, "\n")

#         out_label = f"mocks/pred/grid_{j}"
#         print(out_label)

#         if out_label in f_out:
#             del f_out[out_label]
#         f_out.create_dataset(name=out_label, data=obs_pred)



### galaxy clustering

In [23]:
# def evaluate_mock(f_out, obs_file, obs_label):
#     with h5py.File(obs_file, "r") as f_in:
#         gc_map = []
#         for i in range(1,5):
#             gc_map.append(f_in[f"maglim/galaxy_counts_bin{i}"][:])
#         gc_map = np.stack(gc_map, axis=-1)

#     # for i in range(gc_map.shape[-1]):
#     #     # hp.mollview(gc_map[:,i], title=f"{obs_label} bin {i}")
#     #     hp.gnomview(gc_map[:,i], title=f"{obs_label} bin {i}", rot=(90, -30, 0))

#     obs_map, _, _ = observation.forward_model_observation_map(
#         gc_count_map=gc_map,
#         conf=msfm_conf,
#         apply_norm=True,
#         with_padding=True,
#         nest=False,
#     )

#     obs_pred = model(obs_map[np.newaxis], training=False)
#     obs_pred = np.squeeze(obs_pred)

#     print("\n", obs_label, obs_pred, "\n")

#     out_label = f"mocks/pred/{obs_label}" 
#     if out_label in f_out:
#         del f_out[out_label]
#     f_out.create_dataset(name=out_label, data=obs_pred)


In [24]:
# out_file = os.path.join(base_dir, model_dir, f"preds_{n_steps}.h5")
# with h5py.File(out_file, "a") as f_out:
#     for obs_label in mocks:
#         if obs_label == "Buzzard_flock":
#             buzzard_flock_dir = "/global/u2/a/athomsen/multiprobe-simulation-forward-model/data/mock_observations/Buzzard_flock"
#             buzzard_flock_files = os.listdir(buzzard_flock_dir)
#             for buzzard_flock_file in buzzard_flock_files:
#                 obs_file = os.path.join(buzzard_flock_dir, buzzard_flock_file)
#                 obs_label = buzzard_flock_file[23:-3]
#                 evaluate_mock(f_out, obs_file, obs_label)
#         else:
#             obs_file = f"/global/u2/a/athomsen/multiprobe-simulation-forward-model/data/mock_observations/DESY3_mock_observation_{obs_label}.h5"
#             evaluate_mock(f_out, obs_file, obs_label)

#     print(f"Saved the prediction to {out_file}")

### check against the CosmoGrid internal predictions

In [25]:
# # get cosmogrid cosmologies
# data = h5py.File('/global/cfs/cdirs/des/cosmogrid/CosmoGridV1_metainfo.h5','r')
# cosmo_arr = data['parameters/grid'][()]

# grid_cosmo_paths = [item.astype('str').split('/')[-2] for item in cosmo_arr['path_par']]
# cosmo_params_cosmogridv11 = np.c_[cosmo_arr['Om'].astype('float'),cosmo_arr['s8'].astype('float'),cosmo_arr['w0'].astype('float'),cosmo_arr['wa'].astype('float'),cosmo_arr['Ob'].astype('float'),cosmo_arr['ns'].astype('float'),cosmo_arr['H0'].astype('float')/100]

# # select some cosmologies
# cosmogrid_grid_idx = [0,38,654,710]

# # cosmo_params_cosmogridv11[cosmogrid_grid_idx]

In [26]:
# clustering_file = "/pscratch/sd/j/jbucko/DESY3/v13/linear_bias/clustering/maps/maps_for_compression_v13_clustering_True_lensing_False_grid_cosmo.pkl"
# with open(clustering_file, 'rb') as f:
#     jozef_maps = pickle.load(f)

# n_side = msfm_conf["analysis"]["n_side"]
# n_pix = msfm_conf["analysis"]["n_pix"]
# data_vec_pix = files.load_pixel_file(msfm_conf)[0]
# hp_datapath = "/global/u2/a/athomsen/multiprobe-simulation-forward-model/data/healpy_data"

# with h5py.File(out_file, "a") as f_out:
#     for i in range(jozef_maps.shape[0]):
#         j = cosmogrid_grid_idx[i]
        
#         gc_map = jozef_maps[i]
#         print(gc_map.shape)

#         obs_pred = model(gc_map, training=False)
#         obs_pred = np.squeeze(obs_pred)

#         # print("\n", obs_label, obs_pred, "\n")

#         out_label = f"mocks/pred/grid_{j}" 
#         print(out_label)

#         if out_label in f_out:
#             del f_out[out_label]
#         f_out.create_dataset(name=out_label, data=obs_pred)



In [27]:
# n_noise = 8
# obs_label = "Cardinal"

# out_file = os.path.join(base_dir, model_dir, f"preds_{n_steps}.h5")
# with h5py.File(out_file, "a") as f_out:
#     obs_file = f"/global/u2/a/athomsen/multiprobe-simulation-forward-model/data/mock_observations/DESY3_mock_observation_{obs_label}.h5"

#     for i in range(n_noise):
#         evaluate_mock(f_out, obs_file, f"Cardinal_noise_{i}")

#     print(f"Saved the prediction to {out_file}")

# OLD

In [28]:
# with h5py.File(out_file, "r") as f:
#     fidu_preds = f["fiducial/vali/pred"][:]
    
# print(fidu_preds[:3])

In [29]:
# import pickle

# jozef_dir = "/pscratch/sd/j/jbucko/DESY3/v11/linear_bias/clustering/maps/maps_for_compression_v11_clustering_grid_cosmo_noise_fixed.pkl"
# with open(jozef_dir, "rb") as f:
#     jozef_maps = pickle.load(f)
    

# jozef_preds = []
# for i in range(jozef_maps.shape[0]):
#     jozef_preds.append(model(jozef_maps[i]).numpy())
# jozef_preds = np.stack(jozef_preds, axis=0)

# print(jozef_maps.shape)
# print(jozef_preds.shape)
# np.save("/pscratch/sd/j/jbucko/DESY3/v11/linear_bias/clustering/maps/preds_for_compression_v11_clustering_grid_cosmo_noise_fixed_v2.npy", jozef_preds)

# CosmoGrid

### single realization

In [30]:
# # tomo_bg = [1.3433586, 1.4218498, 1.4996815, 1.5668583]
# tomo_bg = [1.8, 1.9, 2.0, 2.1]

# qbgs = np.round(np.arange(-1, 1.2, 0.2), decimals=1)
# for qbg in tqdm(qbgs):
#     tomo_qbg = 4 * [qbg]
#     obs_label = f"CosmoGrid_qbg={qbg},bg=high"

#     _, gc_count_map = observation.forward_model_cosmogrid(
#         "/global/cfs/cdirs/des/cosmogrid/v11desy3/CosmoGrid/v11desy3/fiducial/cosmo_fiducial/perm_0000",    
#         conf=msfm_conf,
#         noisy=True,
#         with_lensing=False,
#         with_clustering=True,
#         tomo_bg=tomo_bg,
#         tomo_qbg=tomo_qbg
#     )

#     cosmogrid_obs, _, _ = observation.forward_model_observation_map(
#         conf=msfm_conf,
#         gc_count_map=gc_count_map,
#         apply_norm=False,
#         with_padding=True,
#         nest=False,
#     )

#     obs_pred = model(cosmogrid_obs[np.newaxis], training=False)
#     obs_pred = np.squeeze(obs_pred)

#     print("\n", obs_label, obs_pred, "\n")
#     with h5py.File(out_file, "a") as f_out:
#         out_label = f"mocks/pred/{obs_label}" 
#         if out_label in f_out:
#             del f_out[out_label]
#         f_out.create_dataset(name=out_label, data=obs_pred)

### multiple realizations

In [31]:
# tomo_bg = [1.3433586, 1.4218498, 1.4996815, 1.5668583]

# qbg = 1
# tomo_qbg = 4 * [qbg]
# obs_label = f"CosmoGrid_qbg={qbg}"

# # tomo_qbg = None
# # obs_label = f"CosmoGrid_qbg=0"

# obs_preds = []
# with h5py.File(out_file, "a") as f_out:
#     for i in tqdm(range(10)):
#         _, gc_count_map = observation.forward_model_cosmogrid(
#             f"/global/cfs/cdirs/des/cosmogrid/v11desy3/CosmoGrid/v11desy3/fiducial/cosmo_fiducial/perm_{i:04}",    
#             conf=msfm_conf,
#             noisy=True,
#             with_lensing=False,
#             with_clustering=True,
#             tomo_bg=tomo_bg,
#             tomo_qbg=tomo_qbg
#         )

#         cosmogrid_obs, _, _ = observation.forward_model_observation_map(
#             conf=msfm_conf,
#             gc_count_map=gc_count_map,
#             apply_norm=False,
#             with_padding=True,
#             nest=False,
#         )

#         obs_pred = model(cosmogrid_obs[np.newaxis], training=False)
#         obs_pred = np.squeeze(obs_pred)
#         obs_preds.append(obs_pred)

#         # obs_label = f"CosmoGrid_qbg_{i}"
#         # print("\n", obs_label, obs_pred, "\n")

#         out_label = f"mocks/pred/{obs_label}_{i}" 
#         if out_label in f_out:
#             del f_out[out_label]
#         f_out.create_dataset(name=out_label, data=obs_pred)
        
#     obs_preds = np.mean(obs_preds, axis=0)
#     # obs_label = f"CosmoGrid_qbg_mean"
        
#     out_label = f"mocks/pred/{obs_label}_mean" 
#     if out_label in f_out:
#         del f_out[out_label]
#     f_out.create_dataset(name=out_label, data=obs_pred)


# trash

In [32]:
# map_dir = "/global/cfs/cdirs/des/cosmogrid/v11desy3/CosmoGrid/v11desy3/fiducial/cosmo_fiducial/perm_0000"
# conf = files.load_config(
#     "/global/homes/a/athomsen/multiprobe-simulation-forward-model/configs/v11/power_law.yaml"
# )

# wl_gamma_map, gc_count_map = observation.forward_model_fiducial_cosmogrid(
#     map_dir, conf=conf, with_lensing=False, with_clustering=True, noisy=True
# )

# obs_map, _, _ = observation.forward_model_observation_map(
#     gc_count_map=gc_count_map,
#     conf=conf,
#     apply_norm=True,
#     with_padding=True,
#     nest=False,
# )

# # for i in range(obs_map.shape[-1]):
# #     # hp.mollview(gc_map[:,i], title=f"{obs_label} bin {i}")
# #     hp.gnomview(obs_map[:,i], title=f"{obs_label} bin {i}", rot=(90, -30, 0))

# obs_pred = model(obs_map[np.newaxis], training=False)
# obs_pred = np.squeeze(obs_pred)
# print(obs_pred)

In [33]:
# out_file = os.path.join(base_dir, model_dir, f"preds_{n_steps}.h5")
# with h5py.File(out_file, "a") as f_out:
#     for obs_label in mocks:
#         obs_file = f"/global/u2/a/athomsen/multiprobe-simulation-forward-model/data/mock_observations/DESY3_mock_observation_{obs_label}.h5"
#         evaluate_mock(f_out, obs_file, obs_label)
                    
#     print(f"Saved the prediction to {out_file}")

In [34]:
# buzzard_flock_dir = "/global/u2/a/athomsen/multiprobe-simulation-forward-model/data/mock_observations/Buzzard_flock"
# buzzard_flock_files = os.listdir(buzzard_flock_dir)

# with h5py.File(out_file, "a") as f_out:
#     for buzzard_flock_file in buzzard_flock_files:
#         obs_file = os.path.join(buzzard_flock_dir, buzzard_flock_file)
#         obs_label = buzzard_flock_file[23:-3]
#         evaluate_mock(f_out, obs_file, obs_label)
                    
#     print(f"Saved the prediction to {out_file}")

In [35]:
# buzzard_flock_dir = "/global/u2/a/athomsen/multiprobe-simulation-forward-model/data/mock_observations/Buzzard_flock"
# buzzard_flock_files = os.listdir(buzzard_flock_dir)

# with h5py.File(out_file, "a") as f_out:
#     for buzzard_flock_file in buzzard_flock_files:

#         obs_file = os.path.join(buzzard_flock_dir, buzzard_flock_file)
#         obs_label = buzzard_flock_file[23:-3]
        
#         with h5py.File(obs_file, "r") as f_in:
#             gc_map = []
#             for i in range(1,5):
#                 gc_map.append(f_in[f"maglim/galaxy_counts_bin{i}"][:])
#             gc_map = np.stack(gc_map, axis=-1)


In [36]:
# out_file = os.path.join(base_dir, model_dir, f"preds_{n_steps}")
# with h5py.File(out_file, "a") as f_out:
#     for obs_label in mocks:
#         obs_file = f"/global/u2/a/athomsen/multiprobe-simulation-forward-model/data/mock_observations/DESY3_mock_observation_{obs_label}.h5"

# #         with h5py.File(obs_file, "r") as f_in:
# #             gc_map = []
# #             for i in range(1,5):
# #                 gc_map.append(f_in[f"maglim/galaxy_counts_bin{i}"][:])
# #             gc_map = np.stack(gc_map, axis=-1)

# #         # fig, ax = plt.subplots(figsize=(12,6) )
# #         # for i in range(gc_map.shape[-1]):
# #         #     # hp.mollview(gc_map[:,i], title=f"{obs_label} bin {i}")
# #         #     hp.gnomview(gc_map[:,i], title=f"{obs_label} bin {i}", rot=(90, -30, 0))

# #         obs_map, _, _ = observation.forward_model_observation_map(
# #             gc_count_map=gc_map,
# #             conf=msfm_conf,
# #             apply_norm=True,
# #             with_padding=True,
# #             nest=False,
# #         )

# #         obs_pred = model(obs_map[np.newaxis], training=False)
# #         obs_pred = np.squeeze(obs_pred)

# #         print("\n", obs_label, obs_pred, "\n")

# #         out_label = f"mocks/{obs_label}" 
# #         if out_label in f_out:
# #             del f_out[out_label]
# #         f_out.create_dataset(name=out_label, data=obs_pred)
                    
#     print(f"Saved the prediction to {out_file}")

# evaluate the network

In [37]:
# obs_file = "/global/u2/a/athomsen/multiprobe-simulation-forward-model/data/mock_observations/DESY3_mock_observation_Buzzard.h5"
# obs_label = f"Buzzard"

# # obs_file = "/global/u2/a/athomsen/multiprobe-simulation-forward-model/data/mock_observations/DESY3_Cardinal_mock.h5"
# # obs_label = f"Cardinal"

# with h5py.File(obs_file, "r") as f:
#     gc_map = []
#     for i in range(1,5):
#         gc_map.append(f[f"maglim/galaxy_counts_bin{i}"][:])
#     gc_map = np.stack(gc_map, axis=-1)
        
# # for i in range(gc_map.shape[-1]):
# #     hp.mollview(gc_map[:,i], title=f"bin {i}")
# #     hp.gnomview(gc_map[:,i], title=f"bin {i}", rot=(90, -30, 0))

# obs_map, _, _ = observation.forward_model_observation_map(
#     gc_count_map=gc_map,
#     conf=msfm_conf,
#     apply_norm=True,
#     with_padding=True,
#     nest=False,
# )

In [38]:
# obs_pred = model(obs_map[np.newaxis], training=False)
# obs_pred = np.squeeze(obs_pred)

# print(obs_pred)

# out_file = os.path.join(base_dir, model_dir, "mock_preds.h5")
# with h5py.File(out_file, "w") as f:
#     f.create_dataset(obs_label, data=obs_pred)
#     print(f"Saved the prediction to {out_file}")

In [39]:
# for _ in range(3):
#     print(model(obs_map[np.newaxis], training=False))
    
# print("\n")
    
# for _ in range(3):
#     print(model(obs_map[np.newaxis], training=True))

# dropout debug 

In [40]:
# from deep_lss.nets.regression_head import get_regression_head

# feature_dim = 10
# example = tf.random.uniform((1,feature_dim))

# layers = []
# layers.append(tf.keras.layers.Input(shape=(feature_dim,)))
# layers.append(tf.keras.layers.Dense(32))
# layers.extend(get_regression_head(feature_dim, head_type="dense", dropout_rate=0.1))

In [41]:
# model = tf.keras.models.Sequential(layers)

# for _ in range(3):
#     print(model(example))
    
# print("\n")
    
# for _ in range(3):
#     print(model(example, training=True))

In [42]:
# from deepsphere.healpy_networks import HealpyGCNN

# hp_model = HealpyGCNN(
#     nside=512,
#     indices=np.arange(10),
#     layers=layers
# )

# for _ in range(3):
#     print(hp_model(example))
    
# print("\n")

# for _ in range(3):
#     print(hp_model(example, training=True))

In [43]:
# base_model = BaseModel(
#     network=layers,
# )

# for _ in range(3):
#     print(base_model(example))
    
# print("\n")

# for _ in range(3):
#     print(base_model(example, training=True))

In [44]:
# base_model = BaseModel(
#     network=layers,
#     n_side=512,
#     indices=np.arange(10),
# )

# for _ in range(3):
#     print(base_model(example))
    
# print("\n")

# for _ in range(3):
#     print(base_model(example, training=True))

In [45]:
# model = BaseModel(
#     network=network,
#     n_side=msfm_conf["analysis"]["n_side"],
#     indices=data_vec_pix,
#     n_neighbors=net_conf["network"]["n_neighbors"],
#     input_shape=(None, len(data_vec_pix), n_z),
#     strategy=None,
# )

# for _ in range(3):
#     print(model(obs_map[np.newaxis], training=False))
    
# print("\n")
    
# for _ in range(3):
#     print(model(obs_map[np.newaxis], training=True))